# 02 - Model Selection and Training
### DI Leather · Forecasting weekly production output (`actual_qty`)

| | |
|---|---|
| **Input** | `final_features.csv` (from Notebook 1) |
| **Output** | `production_model.joblib` (the full preprocessing + model pipeline) |
| **Next notebook** | `03_Prediction.ipynb` |

**Part A - Model selection:** train seven regression models on the earlier 80% of weeks, test them on the most recent 20%, and pick the best.

**Part B - Final training:** retrain the chosen model on all weeks and save it.


---
# Part A - Model Selection
## Step 1 - Imports

In [1]:
# DataFrame.to_markdown() needs the 'tabulate' package:  pip install tabulate

import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

## Step 2 - Load the feature table
The file must already be in chronological order; the check below stops the notebook if it is not.

In [2]:
df = pd.read_csv("final_features.csv", parse_dates=['week_start_date'])

assert df['week_start_date'].is_monotonic_increasing, "Data must be sorted by week_start_date"

print("Dataset shape:", df.shape)

Dataset shape: (9495, 18)


## Step 3 - Features, target and chronological 80/20 split
- **Target:** `actual_qty`.
- **Features:** every column except the target and the `week_start_date` ordering key.
- **Split:** the first 80% of rows (earlier weeks) train the model; the last 20% (most recent weeks) test it. There is **no random shuffling** — a random split would let the model see future weeks during training and give over-optimistic results.


In [3]:
target = 'actual_qty'

X_full = df.drop(columns=[target, 'week_start_date'])
y_full = df[target]

split_idx = int(len(df) * 0.8)
X_train, X_test = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

print(f"Train: {len(X_train):,} rows  ({df['week_start_date'].iloc[0].date()} to {df['week_start_date'].iloc[split_idx - 1].date()})")
print(f"Test : {len(X_test):,} rows  ({df['week_start_date'].iloc[split_idx].date()} to {df['week_start_date'].iloc[-1].date()})")

Train: 7,596 rows  (2022-04-18 to 2025-03-24)
Test : 1,899 rows  (2025-03-24 to 2026-03-30)


**How to read this:** the test period must start *after* the training period ends. The model is therefore judged on weeks it has never seen, the same situation it faces when forecasting real future production.


## Step 4 - Preprocessing pipeline
Each group of columns is transformed differently. The transformers live **inside** the pipeline, so they are fitted on training rows only when the pipeline is trained.

| Transformer | Columns | What it does |
|---|---|---|
| `StandardScaler` | numeric columns | Rescales each to mean 0, standard deviation 1 |
| `OneHotEncoder` | `financial_month`, `factory_unit`, `product_family`, `grade`, `order_category`, `size_id` | One 0/1 column per category; unseen categories become all zeros |
| `TargetEncoder` | `item_name` | Replaces each item by a smoothed average of `actual_qty` for that item; unseen items get the overall average |
| `passthrough` | `bottleneck_flag` | Already 0/1, so it is passed on unchanged |


In [4]:
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
binary_cols = ['bottleneck_flag']
numerical_cols = [col for col in X_train.columns if col not in categorical_ohe + categorical_target + binary_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target),
        ('binary', 'passthrough', binary_cols)
    ])

print("Numeric columns:", numerical_cols)
print("Binary columns :", binary_cols)

Numeric columns: ['week_no', 'week_of_month', 'rate', 'no_of_workers', 'working_hours', 'machine_count', 'machine_worker_ratio', 'avg_hours_per_worker']
Binary columns : ['bottleneck_flag']


## Step 5 - Candidate models
Seven regressors covering three families:

| Family | Models |
|---|---|
| Linear | Linear Regression, Ridge, Lasso |
| Bagged trees | Random Forest, Extra Trees |
| Boosted trees | Gradient Boosting, XGBoost |


In [5]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}

## Step 6 - Train and evaluate every model
Each model is wrapped in the same pipeline (preprocessing + model), trained on the training weeks, and scored on **both** the training and the test weeks. Comparing the two reveals overfitting.

Metrics: **R²** (share of variation explained, higher is better), **MAE** (average error in shoes) and **RMSE** (error in shoes, penalising large misses more).

In [6]:
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', clone(preprocessor)), ('model', model)])

    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Train R²': round(r2_score(y_train, y_train_pred), 3),
        'Test R²': round(r2_score(y_test, y_test_pred), 3),
        'Train MAE': round(mean_absolute_error(y_train, y_train_pred), 3),
        'Test MAE': round(mean_absolute_error(y_test, y_test_pred), 3),
        'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 3),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 3)
    })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

| Model             |   Train R² |   Test R² |   Train MAE |   Test MAE |   Train RMSE |   Test RMSE |
|:------------------|-----------:|----------:|------------:|-----------:|-------------:|------------:|
| Linear Regression |      0.803 |     0.783 |       4.465 |      4.721 |        5.784 |       6.108 |
| Ridge Regression  |      0.803 |     0.783 |       4.466 |      4.715 |        5.786 |       6.107 |
| Lasso Regression  |      0.798 |     0.779 |       4.507 |      4.759 |        5.851 |       6.171 |
| Random Forest     |      0.935 |     0.766 |       2.441 |      4.831 |        3.312 |       6.344 |
| Gradient Boosting |      0.825 |     0.791 |       4.216 |      4.577 |        5.454 |       6     |
| Extra Trees       |      0.987 |     0.757 |       0.842 |      4.919 |        1.491 |       6.475 |
| XGBoost           |      0.919 |     0.763 |       2.865 |      4.845 |        3.718 |       6.389 |


### How to read the results
- **Test R²** (higher is better) and **Test MAE / RMSE** (lower is better, in shoes) show how well each model forecasts weeks it has never seen.
- **Train vs test gap:** similar values mean the model generalises. Train far above test means **overfitting** - the model memorised the training weeks. Fully grown tree models (Random Forest, Extra Trees) and XGBoost are the most prone to this.
- **Linear vs tree models:** if the linear models score close to the tree models, the inputs relate to output mostly additively and extra complexity buys little.
- **Size of the error:** divide Test MAE by the average `actual_qty` (about 37 shoes per row in the EDA) to express the typical error as a percentage.


## Step 7 - Rank the models and select the best
Models are ranked by **test R²**. The train–test gap is added as an overfitting check; the top-ranked model is selected.

In [7]:
results_df['Gap (Train−Test R²)'] = (results_df['Train R²'] - results_df['Test R²']).round(3)

ranked = results_df.sort_values('Test R²', ascending=False).reset_index(drop=True)
print(ranked[['Model', 'Test R²', 'Test MAE', 'Test RMSE', 'Gap (Train−Test R²)']].to_markdown(index=False))

best = ranked.iloc[0]
print("\nSelected model:", best['Model'])
print("Test R²       :", best['Test R²'])
print("Test MAE      :", best['Test MAE'])
print("Test RMSE     :", best['Test RMSE'])

| Model             |   Test R² |   Test MAE |   Test RMSE |   Gap (Train−Test R²) |
|:------------------|----------:|-----------:|------------:|----------------------:|
| Gradient Boosting |     0.791 |      4.577 |       6     |                 0.034 |
| Linear Regression |     0.783 |      4.721 |       6.108 |                 0.02  |
| Ridge Regression  |     0.783 |      4.715 |       6.107 |                 0.02  |
| Lasso Regression  |     0.779 |      4.759 |       6.171 |                 0.019 |
| Random Forest     |     0.766 |      4.831 |       6.344 |                 0.169 |
| XGBoost           |     0.763 |      4.845 |       6.389 |                 0.156 |
| Extra Trees       |     0.757 |      4.919 |       6.475 |                 0.23  |

Selected model: Gradient Boosting
Test R²       : 0.791
Test MAE      : 4.577
Test RMSE     : 6.0


### Justifying the choice
- **Selected:** the model with the highest test R² (it also has the lowest test MAE/RMSE if the ranking is consistent). Check that its train–test gap is small.
- **Rejected for overfitting:** models whose train R² is far above their test R² (a large `Gap`) - they will forecast new weeks less reliably than their training score suggests.
- **Rejected for accuracy:** models that are stable but score lower on the test weeks.

Note that the models are compared with default settings, without hyperparameter tuning, and that near-ties (differences of about 0.005 in R²) are within normal run-to-run noise.


---
# Part B - Final Training and Saving
## Step 8 - Retrain the selected model on all weeks
Part A used 80% of the weeks only so that the model could be tested honestly. For the model that will actually be used, it is retrained on **all** weeks, so it also learns from the most recent production patterns. The test scores from Part A remain the best estimate of its accuracy - a model trained on all weeks cannot be tested on data it has not seen.

The model chosen in Step 7 is used automatically. The saved file contains the **whole pipeline** (preprocessing + model), so new data can be passed in directly.

In [8]:
best_name = best['Model']

final_pipeline = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('model', clone(models[best_name]))
])

print(f"Training final model: {best_name}...")
final_pipeline.fit(X_full, y_full)

model_filename = 'production_model.joblib'
joblib.dump(final_pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Training final model: Gradient Boosting...
Model successfully saved to production_model.joblib


## Step 9 - Check that the saved model loads and predicts
The file is reloaded and used on the last five rows. These weeks were part of the training data, so this only confirms the saved file works - it is **not** an accuracy test.

In [9]:
loaded_model = joblib.load('production_model.joblib')

sample = X_full.tail(5)
pd.DataFrame({
    'actual_qty': y_full.tail(5).values,
    'predicted_qty': loaded_model.predict(sample).round().astype(int)
})

,actual_qty,predicted_qty
0,27,25
1,22,27
2,33,36
3,25,28
4,24,30


---
## Summary
- The selected model and its test R², MAE and RMSE are shown in Step 7.
- Saved pipeline: `production_model.joblib` - used by `03_Prediction.ipynb`.
